# Boltz-2 Structure Prediction

This notebook demonstrates multi-modal structure prediction using Boltz-2, an open-source model developed by MIT. Boltz-2 predicts the three-dimensional structures of proteins, nucleic acids (DNA and RNA), small-molecule ligands, and their complexes using a diffusion-based generative approach with configurable recycling and sampling steps. We demonstrate `run_boltz2` by predicting the structure of the MfnG protein bound to L-tyrosine, covering input construction, model configuration, result inspection, visualization, and export.

In [1]:
from proto_tools.utils.notebook_docs import display_overview, display_api_reference, display_docs_section, display_doc_link, display_available_tools
display_doc_link("boltz2")
display_overview("boltz2")
display_docs_section("boltz2", "Background")

# Boltz2

> Boltz2 is a state-of-the-art multi-modal structure prediction model that predicts 3D structures of proteins, DNA, RNA, and ligands using a [diffusion](https://en.wikipedia.org/wiki/Diffusion_model)-based architecture with MSA integration. It provides the broadest molecular type support among current structure prediction tools.

## Background

**What does this tool measure/predict?**
Boltz2 predicts the 3D atomic coordinates of biomolecular complexes involving proteins, nucleic acids (DNA/RNA), and small molecule ligands. It outputs full-atom structures with comprehensive confidence metrics including pTM, ipTM, pLDDT, and specialized interface scores for different molecular type combinations.

**Why is this important?**
The ability to model protein-nucleic acid and multi-molecular complexes is essential for:

* Understanding gene regulation (transcription factor-DNA complexes)
* Modeling RNA-protein interactions (ribonucleoproteins, RNA-binding proteins)
* Drug design targeting nucleic acids (RNA therapeutics, [antisense oligonucleotides](https://en.wikipedia.org/wiki/Antisense_therapy))
* [CRISPR](https://en.wikipedia.org/wiki/CRISPR) and gene editing tool development (Cas protein-guide RNA-DNA complexes)
* Studying ribosomes, spliceosomes, and other RNA-protein machines

**Scientific foundation:**
Boltz2 combines several modern deep learning advances:

1. **MSA-based evolutionary features**: [Multiple sequence alignments](https://en.wikipedia.org/wiki/Multiple_sequence_alignment) provide critical evolutionary co-variation signals that indicate residue contacts and structural constraints.
2. **Diffusion-based generative modeling**: Starting from noise, the model iteratively refines coordinates through learned denoising steps, naturally handling the flexibility of biomolecular complexes.
3. **Multi-modal architecture**: Specialized encoders for proteins (amino acid sequences), DNA/RNA (nucleotide sequences), and ligands ([SMILES](https://en.wikipedia.org/wiki/Simplified_molecular-input_line-entry_system)) enable unified prediction of heterogeneous complexes.
4. **Extensive sampling**: By default generates 25 independent structure samples and returns the best by confidence, exploring conformational diversity.

Confidence metrics include:

* **pTM** (predicted TM-score): Global structure accuracy (0-1), primary metric for single chains.
* **ipTM** (interface pTM): Confidence in inter-chain interfaces (0-1), primary metric for complexes.
* **pLDDT** (predicted LDDT): Per-residue confidence scores.
* **Specialized interface scores**: `ligand_iptm`, `protein_iptm` for specific interaction types.

## Available tools

In [2]:
display_available_tools("boltz2")

- **`run_boltz2()`** — Multi-modal structure prediction using Boltz2

### `run_boltz2`

Boltz-2 predicts 3D structures of proteins, DNA, RNA, ligands, and their complexes using a diffusion-based generative model. It supports both single-sequence mode and MSA-assisted prediction via ColabFold search. The `recycling_steps` parameter controls iterative structural refinement, while `sampling_steps` governs the granularity of the denoising process. When `diffusion_samples` is set above 1, Boltz-2 generates multiple independent structure samples and returns the best by confidence score, which is particularly useful for exploring conformational diversity in flexible protein-ligand systems. Ligands are provided as SMILES strings and are automatically converted to the appropriate internal representation.

In [3]:
from pathlib import Path

from proto_tools import (
    Boltz2Config,
    Boltz2Input,
    Chain,
    StructurePredictionComplex,
    run_boltz2,
)

In [4]:
# Display input docs
display_api_reference("boltz2", "input", "run_boltz2")

# MfnG protein sequence (enzyme from methanogenic archaea)
mfng_sequence = "MVTPEGNVSLVDESLLVGVTDEDRAVRSAHQFYERLIGLWAPAVMEAAHELGVFAALAEAPADSGELARRLDCDARAMRVLLDALYAYDVIDRIHDTNGFRYLLSAEARECLLPGTLFSLVGKFMHDINVAWPAWRNLAEVVRHGARDTSGAESPNGIAQEDYESLVGGINFWAPPIVTTLSRKLRASGRSGDATASVLDVGCGTGLYSQLLLREFPRWTATGLDVERIATLANAQALRLGVEERFATRAGDFWRGGWGTGYDLVLFANIFHLQTPASAVRLMRHAAACLAPDGLVAVVDQIVDADREPKTPQDRFALLFAASMTNTGGGDAYTFQEYEEWFTAAGLQRIETLDTPMHRILLARRATEPSAVPEGQASENLYFQ"

# L-tyrosine SMILES
tyrosine_smiles = "c1cc(ccc1C[C@@H](C(=O)O)N)O"

# Create protein-ligand complex
complex = StructurePredictionComplex(
    chains=[
        Chain(sequence=mfng_sequence, entity_type="protein"),
        Chain(sequence=tyrosine_smiles, entity_type="ligand"),
    ]
)

# Create input
inputs = Boltz2Input(complexes=[complex])

**Input** — `Boltz2Input`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `complexes` | `List[StructurePredictionComplex]` | required | List of complexes to predict structures for. Inherited from `StructurePredictionInput`. Each complex can contain multiple chains of proteins, DNA, RNA, and/or ligands. |
| `chains` | `List[Chain]` | required | Chains in the complex. Each chain can be: |
| `msas` | `Dict[string, MSA]` |  | Pre-computed MSAs keyed by protein sequence. Populated by preprocess() or supplied directly. Default: None. |

In [5]:
# Display config docs
display_api_reference("boltz2", "config", "run_boltz2")

# Configure Boltz-2 with reduced settings for a fast demonstration run
config = Boltz2Config(
    verbose=False,
    device="cuda",  # Change to "cpu" if no GPU available
    use_msa=True,
    recycling_steps=3,
    diffusion_samples=1,
)

**Config** — `Boltz2Config`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `recycling_steps` | `integer` | `10` | Number of iterative refinement passes through the model. Higher values produce more refined structures but increase computation time. Typical range: 3-20. Must be at least 0. |
| `sampling_steps` | `integer` | `200` | Number of denoising steps in the diffusion process. Higher values produce more refined structures but are slower. Typical |
| `diffusion_samples` | `integer` | `25` | Number of independent structure samples to generate per complex. Only the best sample (by confidence score) is returned. Higher values explore more of the conformational space but increase computation time. Must be at least 1. Default: 25. |
| `seed` | `integer` |  | Random seed for reproducible results. When set, same seed + same input produces identical output. When None, a random seed is auto-assigned internally via the `resolved_seed` property. Some GPU tools produce approximately (not bit-exact) identical results because CUDA operations reduce floating-point values in non-deterministic thread order. Discrete outputs (sequences, structures) are exact; floating-point outputs (scores, coordinates) may differ at the bit level (\~1e-7). |
| `use_msa` | `boolean` | `True` | Whether to generate and use Multiple Sequence Alignments (MSAs) for protein chains using ColabFold search. Inherited from `MSAStructurePredictionConfig`. Default: `True`. |
| `num_workers` | `integer` | `2` | Number of CPU workers for parallel processing during prediction. Automatically set to the minimum of available CPU cores or 4. Must be at least 1. Default: `min(cpu_count, 4)`. |
| `verbose` | `boolean` | `False` | Whether to print status messages during execution including MSA generation, model loading, and prediction progress. Inherited from `StructurePredictionConfig`. Default: `False`. |
| `device` | `string` | `cuda` | Device to run the model on. Options include `"cuda"` (NVIDIA GPU), `"cpu"` (CPU execution), or specific GPU devices like `"cuda:0"`. Structure prediction is computationally intensive and strongly benefits from GPU acceleration. Default: `"cuda"`. |
| `timeout` | `integer` | `1200` | Maximum execution time in seconds. Default: 1200. |
| `colabfold_search_config` | `ColabfoldSearchConfig` |  | Configuration for ColabFold MSA search. Only used when `use_msa=True`. Inherited from `MSAStructurePredictionConfig`. Default: `None`. |

In [6]:
# Run structure prediction
result = run_boltz2(inputs, config)

Running run_colabfold_search [00:00]

Folding structures (Boltz-2):   0%|          | 0/1 [00:00<?, ?complex/s]

In [7]:
# Display output docs
display_api_reference("boltz2", "output", "run_boltz2")

mfng_structure = result.structures[0]

# Print confidence metrics
print(f"  Number of chains:  {len(complex.chains)}")
print(f"  Protein length:    {len(mfng_sequence)} residues")
print(f"  Confidence score:  {mfng_structure.metrics.confidence_score:.3f}")
print(f"  Complex pLDDT:     {mfng_structure.metrics.complex_plddt:.3f}")
print(f"  pTM score:         {mfng_structure.metrics.ptm:.3f}")
print(f"  ipTM score:        {mfng_structure.metrics.iptm:.3f}")
print(f"  Ligand ipTM:       {mfng_structure.metrics.ligand_iptm:.3f}")

**Output** — `Boltz2Output`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `structures` | `List[Structure]` | required | Predicted structures with confidence metrics. |
| `structure` | `string` | required | Raw structure content in PDB or CIF format. |
| `structure_format` | `string` |  | Format of the content string (auto-detected if omitted). |
| `b_factor_type` | `BFactorType` |  | What the B-factor column represents. |
| `source` | `string` |  | Optional source identifier (filepath or tool name). |
| `metrics` | `StructureMetrics` |  | Associated metrics (e.g., pLDDT, pTM scores, per-chain lists, pairwise matrices). None values are stripped at construction. |

  Number of chains:  2
  Protein length:    384 residues
  Confidence score:  0.910
  Complex pLDDT:     0.916
  pTM score:         0.938
  ipTM score:        0.887
  Ligand ipTM:       0.887


#### Visualize the predicted complex

The interactive viewer renders the predicted protein-ligand complex colored by chain, allowing you to inspect the binding pose and overall fold geometry.

In [8]:
mfng_structure.visualize(style='stick', color_by='chain')

## Export Results

Predicted structures can be exported to PDB or mmCIF format for downstream analysis in molecular visualization tools such as PyMOL, ChimeraX, or VMD. The B-factor column contains pLDDT confidence scores for per-residue visualization.

In [9]:
# Create output directory
output_dir = Path("./example_output")
output_dir.mkdir(exist_ok=True)

# Export results to PDB files
result.export(name="mfng_ligand_complex", export_path=output_dir, file_format="pdb")
print(f"Structure exported to {output_dir / 'mfng_ligand_complex.pdb'}")

Structure exported to example_output/mfng_ligand_complex.pdb


/large_storage/hielab/bviggiano/codebases/evo-design/proto-tools/proto_tools/entities/structures/structure.py:300: UserWarning: CIF→PDB conversion: 1 residue name(s) exceed PDB's 3-character limit and will be truncated (e.g., ['LIG1']).
  return convert_cif_str_to_pdb_str(self.structure)
